In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window as W

In [0]:
PRODUCT_GOLD_COLUMNS = [
    "product_id",
    "name",
    "days_to_manufacture",
    "product_subcategory",
    "product_category"
]

In [0]:
CATALOG_SILVER = 'mini_project'
SCHEMA_SILVER = 'silver_layer'

CATALOG_GOLD = 'mini_project'
SCHEMA_GOLD = 'gold_layer'

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_GOLD}.{SCHEMA_GOLD}")

In [0]:
product_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product")
product_subcategory_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_subcategory")
product_category_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.product_category")

In [0]:
p  = product_df.alias("p")
psc = product_subcategory_df.alias("psc")
pc  = product_category_df.alias("pc")

product_df = (
    p
    .join(
        psc.select(
            F.col("product_subcategory_id"),
            F.col("product_category_id"),
            F.col("name").alias("product_subcategory")
        ),
        on="product_subcategory_id",
        how="left"
    )
    .join(
        pc.select(
            F.col("product_category_id"),
            F.col("name").alias("product_category")
        ),
        on="product_category_id",
        how="left"
    )
)

In [0]:
(
    product_df.select([
        F.col(i) for i in PRODUCT_GOLD_COLUMNS
    ]).write
    .mode("overwrite").saveAsTable(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.product")
)

In [0]:
sales_order_detail_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_detail")
sales_order_header_df = spark.read.table(f"{CATALOG_SILVER}.{SCHEMA_SILVER}.sales_order_header")

In [0]:
display(sales_order_detail_df)

In [0]:
gold_product_df = spark.read.table(f"{CATALOG_GOLD}.{SCHEMA_GOLD}.product")